# Jet Tagging
In this project, you will be classifying the quark/gluon origins of collimated sprays of
particle (jets) in data from the ATLAS detector.
Jets are created from standalone quarks or gluons that can be produced in high energy collisions
due to the creation of new particle anti-particle pairs around these strongly interacting particles.
When short-lived particles are produced in a collision that decay into quarks/gluons, the resulting
jets are collimated themselves and form even larger particle sprays, which can also be called jets.
The dataset for this project describes such large jets which have been recorded by the ATLAS detector.
Each jet in the dataset comes from one of three particles:
1. A quark or a gluon only
2. A W- or a Z-boson decaying into two quarks
3. A top quark decaying into a b-quark and into a W-boson, which further decays into two quarks

## Getting Started
To get started, put the jettagging folder you just unzipped into your `exercises-pai` folder
next to exercises of the last couple weeks and navigate to the base directory of this project
with `cd jettagging`.
Next, run `uv init --lib --no-workspace` to initialize the python module we will use.
This time we want `--no-workspace` to make this project standalone, since you will also
submit it standalone.
You are free to use the python module you just initialized as much as you want, your grade
will ultimately depend on readability, documentation and reproducilibity
of your code, regardless of which specific structure you use.
But in terms of reproducilibity the module is important, as it tracks the dependencies
you use, even if you want to write everything else in a notebook.

### Tracking your Dependencies
When you share your code with us, you want to make sure that we can run it without having
to manually install every package you use.
One way of doing this is to simply remember to run `uv add <package-name>` every time you
use a particular package, i.e. when you import it.
And while we are on it, you can start by adding the dependencies that you are for sure
gonna need for this notebook and the project:

In [ ]:
%%bash
uv add torch numpy matplotlib h5py pandas tables notebook

That being said, this is a fine approach, but you risk missing certain packages that are
needed to run your code, because you already installed them before without listing them.

The other approach is to run `uv sync` at some point, which makes sure that only the packages
you have declared in your standalone `pyproject.toml` for this project are installed.
Specifically this means that it uninstalls any packages that you didn't declare, i.e. your
code won't run if you didn't declare an essential package.
You don't have to do this though, if you feel more comfortable with the approach above.

## Downloading and Inspecting the Data
We are hosting the dataset for this project on `drive.switch.ch`, from which we can directly
download it in python using the requests module.
We are deliberately using code that caches intermediate results, so it can be run repeatedly,
always creating the same output, without doing additional work:

In [ ]:
import requests
from pathlib import Path
from zipfile import ZipFile

CACHE_PATH = Path(".cache/jet_tagging.zip")
DATA_DIR_PATH = Path("data")

if not CACHE_PATH.exists():
    DOWNLOAD_URL = "https://drive.switch.ch/index.php/s/JpqDntLTwRKMgqW/download"
    print(f"downloading jet tagging data from '{DOWNLOAD_URL}'")
    response = requests.get(DOWNLOAD_URL)

    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with CACHE_PATH.open("wb") as jet_tagging_archive:
        jet_tagging_archive.write(response.content)

if not DATA_DIR_PATH.exists():
    print("extracting archive")
    DATA_DIR_PATH.mkdir(parents=True, exist_ok=True)
    with ZipFile(CACHE_PATH) as jet_tagging_archive:
        jet_tagging_archive.extractall(DATA_DIR_PATH)

    (DATA_DIR_PATH / "qcd_const.h5").rename(DATA_DIR_PATH / "jet_constituents_quark_gluon.h5")
    (DATA_DIR_PATH / "wz_const.h5").rename(DATA_DIR_PATH / "jet_constituents_wz.h5")
    (DATA_DIR_PATH / "top_const.h5").rename(DATA_DIR_PATH / "jet_constituents_top.h5")
    (DATA_DIR_PATH / "qcd_small.h5").rename(DATA_DIR_PATH / "jet_properties_global_quark_gluon.h5")
    (DATA_DIR_PATH / "wz_small.h5").rename(DATA_DIR_PATH / "jet_properties_global_wz.h5")
    (DATA_DIR_PATH / "top_small.h5").rename(DATA_DIR_PATH / "jet_properties_global_top.h5")

Now we should have six files in the `data/` folder `jet_constituents_quark_gluon.h5`, `jet_constituents_wz.h5`
and `jet_constituents_top.h5` as well as `jet_properties_global_quark_gluon.h5`, `jet_properties_global_wz.h5` and
`jet_properties_global_top.h5`.

The last three files contain global jet properties, including it's global momentum and
"substructure"-variables, i.e. variables that describe other properties of the jet that are not related
to it's global momentum.
These ones we have to load with pandas (they are saved under the "substructure" key within the file),
let's have a look:

In [ ]:
import h5py
import pandas
import numpy
from typing import cast

jet_properties_global_quark_gluon = cast(pandas.DataFrame, pandas.read_hdf(DATA_DIR_PATH / "jet_properties_global_quark_gluon.h5", key="substructure"))

Tthe first four variables, these describe the global jet momentum, i.e. the sum of the momenta
of its constituents (more on that later).
The next three variables $\tau_N$ are the $N$-subjettiness variables, which try to describe how well the jet
matches an $N$-prong jet.
Classically the ratios $\tau_3 / \tau_2$ and $\tau_2 / \tau_1$ specifically are used to find top jets
and $W$/$Z$-jets, for which the respective value is usually the smallest.
Then we also have the splitting scales $d_{12}$ and $d_{23}$ as well as the energy correlation functions
`ECF2` and `ECF3` which we are not going to explain in detail here, but you are welcome to read up on them, or
ask your favorite LLM about them.
Note that the jet momentum variables have units of `GeV` or are unitless, the $N$-subjetiness variables are
unitless and everything else has units of $\text{GeV}^2$.

As an example here, let's look the $p_T$ distribution over all quark gluon jets:

In [ ]:
from matplotlib import pyplot

pyplot.title("quark gluon jets")
pyplot.hist(jet_properties_global_quark_gluon["pt"], histtype="step")
pyplot.xlabel("p_T [GeV]")
pyplot.ylabel("count")

The first three files on the other hand contain the four-momenta of the jet constituents (of the individual
particles that make up the jet), ordered by their transverse momentum $p_T$, for each associated type of jet.
These are parameterized by the transverse momentum ($p_T$), the pseudorapidity ($eta$),
the polar angle ($phi$) and the mass of the jet ($m$), in that exact order.
Here the transverse momentum describes the momentum directed towards the cylindrical detector, as opposed to
parallel to the beam line, while the pseudorapidity

$$ \eta = -\ln(\tan \frac{\theta}{2}) $$

is a proxy for the azimuthal angle $\theta$, while differences of $\eta$ (unlike $\theta$) are Lorentz-invariant.
These four quantities are equivalent to the standard representation $(E, p_x, p_y, p_z)$ of a four-momentum and
can be converted to it, using the following formulas

$$ p_x = p_T \cos\phi $$
$$ p_y = p_T \sin\phi $$
$$ p_z = p_T \sinh\eta $$
$$ E = \sqrt{m^2 + p_T^2 \cosh^2\eta} $$

Note that $p_T$ and $m$ are in `\text{GeV}`, while `\theta` and `\phi` are unitless.

Let's have a look with h5py:

In [ ]:
import h5py
import numpy
from pathlib import Path

data_folder = Path("data")


with h5py.File(DATA_DIR_PATH / "jet_constituents_quark_gluon.h5") as f:
    jet_constituents_quark_gluon = numpy.array(f["constituents"])
print(jet_constituents_quark_gluon.shape)

Looks like we have 500000 jets in here, with one four-momenta stored for every one of 20 constituents per jet.
Let's try to plot the $\eta$, $\phi$ positions of the first jet constituent, using small or large circles
depending on the $p_T$ value.
Let's also show the position and $p_T$ of the full jet for comparison:

In [ ]:
from matplotlib import pyplot

pyplot.title("quark gluon jets, top $p_T$ constituent")
pyplot.scatter(
    jet_properties_global_quark_gluon.eta[0],
    jet_properties_global_quark_gluon.phi[0],
    s=jet_properties_global_quark_gluon.pt[0]
)
pyplot.scatter(
    jet_constituents_quark_gluon[0, :, 1],
    jet_constituents_quark_gluon[0, :, 2],
    s=jet_constituents_quark_gluon[0, :, 0]
)
pyplot.xlabel(r"$\eta$")
pyplot.ylabel(r"$\phi$")

Think of why it makes sense to use $\eta$ and $\phi$ for the position of the constituents here.
The pseudorapidity $\eta$ is essentially a proxy for the position along the beam axis, while
$\phi$ describes the angle in the plane perpendicular of the beam axis.
Together they essentially give you the position on the surface of the cylindrical ATLAS detector
of each jet constituent, while the $p_T$ gives you the momentum perpendicular to that surface.
Think about if this current data representation makes sense for classifying the jet.
Does the label of the jet depend on it's global position?

This hopefully gives you a good idea about how to download and inspect the data.

## Tasks
In this project you will be given 3 pre-defined tasks.
You will be graded on how well you solve them,
on the structure, documentation and reproducilibity of your code
and how well you analyze and present the results of the tasks.
In particular honest analysis of sub-optimal results will give you
a better grade than a sugarcoated description of such.

### Task 1: Binary Classification
Implement data preparation steps and train a binary classification model on prepared quark/gluon vs. W/Z or
top jets, which can reproduce with high accuracy the correct origin of the jet.
The model should work both on quark/gluon vs. W/Z and quark/gluon vs. top jets.
Analyze the performance of the classification model usings metrics of your choice and try to improve sub-optimal
performance as much as possible.

### Task 2: Multi-Class Classification
Implement data preparation steps and train a mutli-class classification model on prepared quark/gluon, W/Z and
top jets, which can reproduce with high accuracy the correct origin of the jet.
Analyze the performance of the classification model usings metrics of your choice and try to improve sub-optimal
performance as much as possible.

### Task 3: Decorrelation
Try to decorrelate the predicted jet origin from the jet's four momentum, i.e. make the classifier ideally only
sensitive to substructure variables, but not to the global four momentum of the jet.
Analyze the how well the model manages to ignore the jet momentum will still achieving good performance on
task 2, using metrics of your choice and improve your method as much as possible.

## Submission
For the submission, we expect you to submit your code and the presentation slides which you will use during the
oral exam in pdf format.

When submitting your code, remember that you are graded on structure, documentation and reproducibility, essentially the same criteria for publishing any kind of code. This means you should have a clear structure and documentation for how the code works, and how to use it to reproduce the results.

For submission please create a folder called `submission`, move all of your relevant code files into that folder
as well as a `presentation.pdf` file containing your presentation slides.
Next, make sure you have done the following beforehand:
- Document code using docstrings for details and a README.md for an introduction and usage/reproduction instructions.
- delete all unnecessary files in `submission`, this include:
    - all data files.
    - the `.git/` folder if it exists.
    - virtual environment folders (generally named `.venv/`). Exercises from previous weeks, only submit code relevant to this project.
    - anything larger than a couple of megabytes, that isn't essential.
    - all plots, unless used as part of the `README.md`.
    - all code that is no longer used.

Finally, convert the `submission` folder into a zip archive (e.g. with `zip -r submission.zip submission` in bash)
called `submission.zip`, double check that everything is in there and upload it on moodle as your final project
submission before the deadline (remember that late submission result in grade reductions, so please submit on time).